# Assignment Case C — Customer RFM Analysis
## Working with PySpark DataFrames for Analytics
**Day 42 | Batch 13 | 03 Mei 2026**

---

| Field | Isi |
|-------|-----|
| **Nama** | *Yustika Indah Maharani* |
| **NIM** | *(tidak dibagikan oleh dibimbing)* |
| **Case** | Case C — Customer RFM Segmentation |
| **Source** | Hive: `adventureworks.*` |
| **Target** | Hive: `adventureworks_curated.fact_customer_rfm` |

---

## 🎯 Business Question

1. **Siapa customer paling berharga** (Champion & Loyal)?
2. **Customer mana yang perlu di-winback** (At Risk & Lost)?
3. **Distribusi segmen** — berapa % customer di tiap segmen?
4. **Rata-rata revenue** per segmen RFM?

## 📐 RFM Framework

```
R — Recency   : Berapa hari sejak order terakhir? (lebih kecil = lebih baik)
F — Frequency : Berapa kali order? (lebih besar = lebih baik)  
M — Monetary  : Berapa total spending? (lebih besar = lebih baik)
```

## 📊 Scoring 1–4 (Quartile-based)

| Dimensi | Score 4 (Terbaik) | Score 1 (Terburuk) |
|---------|-------------------|--------------------|
| **R** | Recency terkecil (baru beli) | Recency terbesar (lama tidak beli) |
| **F** | Frequency terbesar | Frequency terkecil |
| **M** | Monetary terbesar | Monetary terkecil |

> ⚠️ Recency di-score **terbalik**: recency kecil → score R tinggi

## 🏷️ Segmentasi

| Segment | Kondisi |
|---------|--------|
| **Champion** | R ≥ 3 AND F ≥ 3 AND M ≥ 3 |
| **Loyal** | F ≥ 3 AND M ≥ 3 (bukan Champion) |
| **Potential Loyal** | R ≥ 3 AND F >= 2 |
| **At Risk** | R <= 2 AND F >= 3 AND M >= 3 |
| **Lost** | R = 1 AND F <= 2 |
| **New Customer** | R = 4 AND F = 1 |
| **Others** | Semua yang tidak masuk kategori di atas |

## 📋 Kolom Target: `fact_customer_rfm`

| Kolom | Tipe | Keterangan |
|-------|------|------------|
| `customer_id` | INT | ID customer |
| `customer_name` | STRING | Nama customer |
| `territory_name` | STRING | Wilayah customer |
| `last_order_date` | DATE | Tanggal order terakhir |
| `recency_days` | INT | Hari sejak order terakhir |
| `frequency` | INT | Jumlah order |
| `monetary` | DECIMAL | Total spending |
| `r_score` | INT | Recency score (1-4) |
| `f_score` | INT | Frequency score (1-4) |
| `m_score` | INT | Monetary score (1-4) |
| `rfm_score` | STRING | Gabungan R+F+M score, contoh: '344' |
| `rfm_segment` | STRING | Champion/Loyal/At Risk/dll |
| `load_timestamp` | TIMESTAMP | Waktu load |


## 0. Persiapan: Memastikan Buat Hive External Tables untuk RFM

**Tabel yang akan dibuat di Hive (`adventureworks.*`):**
| Hive Table | Sumber PostgreSQL | Catatan |
|---|---|---|
| `fact_sales_orders` | `Sales.SalesOrderHeader` | partisi `order_year`, `order_month` |
| `dim_customer` | `Sales.Customer` | |
| `dim_person` | `Person.Person` | |
| `dim_territory` | `Sales.SalesTerritory` | |

### 0.1 Setup SparkSession JDBC (untuk Extract ke HDFS)

SparkSession ini **tanpa Hive support** — hanya untuk extract data dari PostgreSQL ke HDFS sebagai Parquet. Sehingga perlu dipastikan service sudah berjalan dan juga telah setup HDFS. 

- Jalankan semua service (Hadoop + Hive + Spark):

  `docker-compose up -d`

- Setup HDFS Directories:

  `make setup-hdfs`


In [ ]:
# environment setup & import
import os
os.environ['PYSPARK_SUBMIT_ARGS'] = '--jars /home/jovyan/work/jars/postgresql-42.7.3.jar pyspark-shell'

from pyspark.sql import SparkSession
from pyspark.sql import functions as F_jdbc

# creating spark_jdbc without Hive support
spark_jdbc = SparkSession.builder \
    .master('local[*]') \
    .appName('RFM-to-HDFS') \
    .config('spark.sql.parquet.compression.codec', 'snappy') \
    .getOrCreate()

# JDBC PostgreSQL AdventureWorks configuration
JDBC_URL   = 'jdbc:postgresql://adventureworks-postgres:5432/postgres'
JDBC_PROPS = {"user": "postgres",
              "password": "My_password1",
              "driver": "org.postgresql.Driver"}
HDFS_BASE  = 'hdfs://namenode:9000/datalake/raw'

# extracting data into pyspark
def read_pg(schema, table):
    return spark_jdbc.read.jdbc(
        url=JDBC_URL,
        table=f'"{schema}"."{table}"',
        properties=JDBC_PROPS
    )

print(f'✓ spark_jdbc {spark_jdbc.version} ready')

✓ spark_jdbc 3.5.0 ready


### 0.2 Extract Tabel Dimensi ke HDFS

Tabel dimensi (Customer, Person, SalesTerritory) tanpa partisi.

In [ ]:
# defining the dim tables
dim_tables = [
    ('Sales', 'Customer'),
    ('Person', 'Person'),
    ('Sales', 'SalesTerritory'),
]

# batch ingesting dim_tables to HDFS
for schema, table in dim_tables:
    df = read_pg(schema, table)
    hdfs_path = f'{HDFS_BASE}/{schema}/{table}'
    df.write.mode('overwrite').parquet(hdfs_path)
    print(f'  {schema}.{table:20s} -> {df.count():6,} rows -> HDFS')

print('✓ Dimension tables have been successfully saved!')

  Sales.Customer             -> 19,820 rows -> HDFS
  Person.Person               -> 19,972 rows -> HDFS
  Sales.SalesTerritory       ->     10 rows -> HDFS
✓ Dimension tables have been successfully saved!


### 0.3 Extract Tabel Fakta dengan Partisi

`SalesOrderHeader` adalah fakta utama RFM. Partisi by `order_year` dan `order_month`.

In [ ]:
# reading SalesOrderHeader from PostgreSQL
df_oh = read_pg('Sales', 'SalesOrderHeader')

# adding partition column order_year & order_month
df_oh_part = df_oh \
    .withColumn('order_year',  F_jdbc.year('orderdate')) \
    .withColumn('order_month', F_jdbc.month('orderdate'))

orders_path = f'{HDFS_BASE}/Sales/SalesOrderHeader'

# saving to HDFS (order_year, order_month)
df_oh_part.write.mode('overwrite') \
    .partitionBy('order_year', 'order_month') \
    .parquet(orders_path)

print(f'✓ SalesOrderHeader -> {df_oh.count():,} rows -> HDFS (partitioned by order_year/order_month)')

✓ SalesOrderHeader -> 31,465 rows -> HDFS (partitioned by order_year/order_month)


### 0.4 Setup SparkSession dengan Hive Support

Sebelum menjalankan cell di bawah, perlu dipastikan sudah melakukan poses inisialisasi dan konfigurasi database untuk Hive Metastore dengan cara berikut:

`docker exec adventureworks-postgres psql -U postgres -c "CREATE DATABASE hive_metastore;"`

`docker exec adventureworks-postgres psql -U postgres -c "\l"`

`docker restart hive-metastore`

`docker logs hive-metastore --tail 20`

`docker restart hiveserver2`

In [ ]:
# stopping spark_jdbc to avoid resouse conflicts
spark_jdbc.stop()

from pyspark.sql import SparkSession

# creating spark_hive with hive support
spark_hive = SparkSession.builder \
    .master('local[*]') \
    .appName('RFM-HiveSetup') \
    .config('spark.jars', '/home/jovyan/work/jars/postgresql-42.7.3.jar') \
    .config('hive.metastore.uris', 'thrift://hive-metastore:9083') \
    .config('spark.sql.warehouse.dir', 'hdfs://namenode:9000/user/hive/warehouse') \
    .enableHiveSupport() \
    .getOrCreate()

HDFS_BASE = 'hdfs://namenode:9000/datalake/raw'

# creating adventureworks database if does not exist
spark_hive.sql('CREATE DATABASE IF NOT EXISTS adventureworks')
spark_hive.sql('USE adventureworks')
print(f'✓ spark_hive {spark_hive.version} ready')
spark_hive.sql('SHOW DATABASES').show()


✓ spark_hive 3.5.0 ready
+--------------+
|     namespace|
+--------------+
|adventureworks|
|       default|
+--------------+



### 0.5 Buat Hive External Tables — Tabel Dimensi

Tabel dimensi tanpa partisi → schema otomatis di-infer dari Parquet.

In [ ]:
# helper function: create hive external table from HDFS parquet
def create_external_table(table_name, hdfs_path):
    spark_hive.sql(f'DROP TABLE IF EXISTS adventureworks.{table_name}')
    spark_hive.sql(f"""
        CREATE EXTERNAL TABLE adventureworks.{table_name}
        USING PARQUET
        LOCATION '{hdfs_path}'
    """)
    count = spark_hive.sql(f'SELECT COUNT(*) FROM adventureworks.{table_name}').collect()[0][0]
    print(f'✓ {table_name:35s} -> {count:6,} rows')

# hive table names mapping to HDFS paths
hive_dim_tables = [
    ('dim_customer',  f'{HDFS_BASE}/Sales/Customer'),
    ('dim_person',    f'{HDFS_BASE}/Person/Person'),
    ('dim_territory', f'{HDFS_BASE}/Sales/SalesTerritory'),
]

# batch registering dim tables to hive
for table_name, path in hive_dim_tables:
    create_external_table(table_name, path)

print('✓ Hive dimension tables completed!')

✓ dim_customer                        -> 19,820 rows
✓ dim_person                          -> 19,972 rows
✓ dim_territory                       ->     10 rows
✓ Hive dimension tables completed!


### 0.6 Buat Hive External Table — Tabel Fakta dengan Partisi

Buat `fact_sales_orders` dengan skema eksplisit + `PARTITIONED BY (order_year, order_month)`,
lalu jalankan `MSCK REPAIR TABLE`.

In [ ]:
orders_path = f'{HDFS_BASE}/Sales/SalesOrderHeader'

# dropping table if exist
spark_hive.sql('DROP TABLE IF EXISTS adventureworks.fact_sales_orders')

# creating an external table for fact table
spark_hive.sql(f"""
    CREATE EXTERNAL TABLE adventureworks.fact_sales_orders (
        SalesOrderID           INT,
        OrderDate              TIMESTAMP,
        DueDate                TIMESTAMP,
        ShipDate               TIMESTAMP,
        CustomerID             INT,
        SalesPersonID          INT,
        TerritoryID            INT,
        SubTotal               DECIMAL(19,4),
        TaxAmt                 DECIMAL(19,4),
        Freight                DECIMAL(19,4),
        TotalDue               DECIMAL(19,4),
        Comment                STRING,
        rowguid                STRING,
        ModifiedDate           TIMESTAMP
    )
    PARTITIONED BY (order_year INT, order_month INT)
    STORED AS PARQUET
    LOCATION '{orders_path}'
""")

# detecting partisions by running MSCK REPAIR TABLE
spark_hive.sql('MSCK REPAIR TABLE adventureworks.fact_sales_orders')

# verifying the row count
cnt = spark_hive.sql('SELECT COUNT(*) FROM adventureworks.fact_sales_orders').collect()[0][0]
print(f'  fact_sales_orders -> {cnt:,} rows')

# checking the fact table
spark_hive.sql('SHOW TABLES IN adventureworks').show(20)
spark_hive.stop()
print('✓ Hive External Tables setup completed!')

  fact_sales_orders -> 31,465 rows
+--------------+-----------------+-----------+
|     namespace|        tableName|isTemporary|
+--------------+-----------------+-----------+
|adventureworks|     dim_customer|      false|
|adventureworks|       dim_person|      false|
|adventureworks|    dim_territory|      false|
|adventureworks|fact_sales_orders|      false|
+--------------+-----------------+-----------+

✓ Hive External Tables setup completed!


## 0. Setup SparkSession

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import *

# configuring spark session with hive support
spark = SparkSession.builder \
    .master('local[*]') \
    .appName('CaseC_CustomerRFM') \
    .config('hive.metastore.uris', 'thrift://hive-metastore:9083') \
    .config('spark.sql.warehouse.dir', 'hdfs://namenode:9000/user/hive/warehouse') \
    .enableHiveSupport() \
    .getOrCreate()

# reference date — use max order date as 'today'
REFERENCE_DATE = '2014-07-31'

print(f'Spark {spark.version} ready')
print(f'Reference date: {REFERENCE_DATE}')

Spark 3.5.0 ready
Reference date: 2014-07-31


## 1. Load Data dari Hive

In [ ]:
# loading all required tables
df_orders    = spark.table('adventureworks.fact_sales_orders')
df_customer  = spark.table('adventureworks.dim_customer')
df_person    = spark.table('adventureworks.dim_person')
df_territory = spark.table('adventureworks.dim_territory')

# checking row counts summary
print('Row counts:')
for name, df in [('fact_sales_orders', df_orders), ('dim_customer', df_customer),
                  ('dim_person', df_person), ('dim_territory', df_territory)]:
    print(f'  {name}: {df.count():,}')

Row counts:
  fact_sales_orders: 31,465
  dim_customer: 19,820
  dim_person: 19,972
  dim_territory: 10


## 2. Eksplorasi Data

In [13]:
# - Rentang tanggal order (min & max orderdate)
# - Jumlah customer unik
# - Distribusi total order per customer

print('=== Rentang Tanggal Order ===')
df_orders.select(F.min('orderdate').alias('Min Order Date'),
                 F.max('orderdate').alias('Max Order Date')
                 ).show()

print('\n=== Jumlah Customer Unik ===')
unique_customers = df_orders.select('customerid').distinct().count()
print(f'Total Unique Customers: {unique_customers}')

print('\n=== Distribusi Order Frequency per Customer ===')
df_frequency = df_orders.groupBy('customerid').agg(F.count('customerid').alias('frequency'))
df_frequency.select('frequency').describe().show()

=== Rentang Tanggal Order ===
+-------------------+-------------------+
|     Min Order Date|     Max Order Date|
+-------------------+-------------------+
|2011-05-31 00:00:00|2014-06-30 00:00:00|
+-------------------+-------------------+


=== Jumlah Customer Unik ===
Total Unique Customers: 19119

=== Distribusi Order Frequency per Customer ===
+-------+------------------+
|summary|         frequency|
+-------+------------------+
|  count|             19119|
|   mean|1.6457450703488676|
| stddev|1.4570539244634981|
|    min|                 1|
|    max|                28|
+-------+------------------+



## 3. Hitung R, F, M per Customer

**Tugas:** Dari `fact_sales_orders`, hitung Recency, Frequency, Monetary per customer.

In [ ]:
# Recency   = datediff(reference_date, max(orderdate))  ← hari sejak order terakhir
# Frequency = count(salesorderid)                       ← total order
# Monetary  = round(sum(totaldue), 2)                   ← total spending

# calculating R,F,M
df_rfm_raw = df_orders \
    .groupBy('customerid') \
    .agg(
        F.datediff(F.lit(REFERENCE_DATE), F.max('orderdate')).alias('recency_days'),
        F.count('customerid').alias('frequency'),
        F.round(F.sum('totaldue'), 2).alias('monetary'),
        F.max('orderdate').alias('last_order_date')
    )

# summary statistics and initial data inspection
print(f'Total customers: {df_rfm_raw.count():,}')
df_rfm_raw.describe().show()
df_rfm_raw.orderBy(F.asc('recency_days')).show(10)

Total customers: 19,119
+-------+-----------------+------------------+------------------+--------+
|summary|       customerid|      recency_days|         frequency|monetary|
+-------+-----------------+------------------+------------------+--------+
|  count|            19119|             19119|             19119|       0|
|   mean|          20559.0|221.26748260892307|1.6457450703488676|    NULL|
| stddev|5519.324233998218| 150.4236048991849|1.4570539244634981|    NULL|
|    min|            11000|                31|                 1|    NULL|
|    max|            30118|              1157|                28|    NULL|
+-------+-----------------+------------------+------------------+--------+

+----------+------------+---------+--------+-------------------+
|customerid|recency_days|frequency|monetary|    last_order_date|
+----------+------------+---------+--------+-------------------+
|     14680|          31|        2|    NULL|2014-06-30 00:00:00|
|     24704|          31|        1|    N

## 4. Scoring R, F, M (Quartile-based)

Gunakan `ntile(4)` window function untuk membagi customer menjadi 4 kuartil.

> **Ingat:** R-score **terbalik** — recency kecil = baru beli = score 4 (bagus).

In [ ]:
# Petunjuk:
# - R score: ntile(4) OVER (ORDER BY recency_days DESC)  ← terbalik! desc = kecil dapat rank 1→4
#   tapi kita mau recency kecil = score 4, jadi pakai:
#   ntile(4) OVER (ORDER BY recency_days ASC) lalu (5 - ntile_value)
#   ATAU langsung: ntile(4) OVER (ORDER BY recency_days DESC)
# - F score: ntile(4) OVER (ORDER BY frequency ASC)
# - M score: ntile(4) OVER (ORDER BY monetary ASC)

# defining window partitions for scoring metrics
win_r = Window.orderBy(F.desc('recency_days'))
win_f = Window.orderBy(F.asc('frequency'))
win_m = Window.orderBy(F.asc('monetary'))

# dividing customers base to generate R,F,M scores
df_scored = df_rfm_raw \
    .withColumn('r_raw',  F.ntile(4).over(win_r)) \
    .withColumn('f_score', F.ntile(4).over(win_f)) \
    .withColumn('m_score', F.ntile(4).over(win_m)) \
    .withColumn('r_score', F.lit(5) - F.col('r_raw'))

# combining R,F,M scores into RFM string
df_scored = df_scored.withColumn(
    'rfm_score',
    F.concat(F.col('r_score').cast('string'),
             F.col('f_score').cast('string'),
             F.col('m_score').cast('string'))
)

# showing score distribution
print('=== Score Distribution ===')
df_scored.groupBy('r_score', 'f_score', 'm_score').count().orderBy('r_score','f_score','m_score').show(20)
print(f'Total customers scored: {df_scored.count():,}')

=== Score Distribution ===
+-------+-------+-------+-----+
|r_score|f_score|m_score|count|
+-------+-------+-------+-----+
|      1|      2|      2|  482|
|      1|      3|      3| 2089|
|      1|      4|      4| 2208|
|      2|      2|      2| 2583|
|      2|      3|      3|  131|
|      2|      4|      4| 2066|
|      3|      1|      1| 1241|
|      3|      2|      2| 1715|
|      3|      3|      3| 1494|
|      3|      4|      4|  330|
|      4|      1|      1| 3539|
|      4|      3|      3| 1066|
|      4|      4|      4|  175|
+-------+-------+-------+-----+

Total customers scored: 19,119


## 5. Segmentasi Customer

In [ ]:
# Champion      : R>=3 AND F>=3 AND M>=3
# Loyal         : F>=3 AND M>=3 (bukan Champion)
# Potential Loyal: R>=3 AND F>=2
# At Risk       : R<=2 AND F>=3 AND M>=3
# Lost          : R=1 AND F<=2
# New Customer  : R=4 AND F=1
# Others        : otherwise

# creating shorthand references for R,F,M score
r = F.col('r_score')
f = F.col('f_score')
m = F.col('m_score')

# categorizing customers into segments
df_segmented = df_scored.withColumn(
    'rfm_segment',
    F.when((r >= 3) & (f >= 3) & (m >= 3), 'Champion')
     .when((f >= 3) & (m >= 3), 'Loyal')
     .when((r >= 3) & (f >= 2), 'Potential Loyal')
     .when((r <= 2) & (f >= 3) & (m >= 3), 'At Risk')
     .when((r == 1) & (f <= 2), 'Lost')
     .when((r == 4) & (f == 1), 'New Customer')
     .otherwise('Others')
)

# customer segments summary
print('=== Distribusi Segmen ===')
df_segmented.groupBy('rfm_segment') \
    .agg(
        F.count('customerid').alias('jumlah_customer'),
        F.round(F.avg('monetary'), 2).alias('avg_spending'),
        F.round(F.avg('frequency'), 1).alias('avg_frequency')
    ) \
    .orderBy(F.desc('jumlah_customer')) \
    .show()

=== Distribusi Segmen ===
+---------------+---------------+------------+-------------+
|    rfm_segment|jumlah_customer|avg_spending|avg_frequency|
+---------------+---------------+------------+-------------+
|          Loyal|           6494|        NULL|          2.3|
|         Others|           3824|        NULL|          1.0|
|   New Customer|           3539|        NULL|          1.0|
|       Champion|           3065|        NULL|          2.3|
|Potential Loyal|           1715|        NULL|          1.0|
|           Lost|            482|        NULL|          1.0|
+---------------+---------------+------------+-------------+



## 6. Enrich dengan Data Customer

In [ ]:
# joining dim_customer, dim_person, dim_territory
df_final_rfm = df_segmented \
    .join(df_customer.select('customerid', 'personid', 'territoryid'), 'customerid', 'left') \
    .join(
        df_person.select(
            F.col('businessentityid').alias('personid'),
            F.concat_ws(' ', 'firstname', 'lastname').alias('customer_name')
        ), 'personid', 'left') \
    .join(F.broadcast(
        df_territory.select(
            'territoryid',
            F.col('name').alias('territory_name')
        )), 'territoryid', 'left') \
    .select(
        'customerid', 'customer_name', 'territory_name',
        'last_order_date', 'recency_days', 'frequency', 'monetary',
        'r_score', 'f_score', 'm_score', 'rfm_score', 'rfm_segment'
    ) \
    .withColumn('load_timestamp', F.current_timestamp())

# showing the final table
print(f'Final RFM rows: {df_final_rfm.count():,}')
df_final_rfm.show(10, truncate=False)

Final RFM rows: 19,119
+----------+----------------+--------------+-------------------+------------+---------+--------+-------+-------+-------+---------+------------+--------------------------+
|customerid|customer_name   |territory_name|last_order_date    |recency_days|frequency|monetary|r_score|f_score|m_score|rfm_score|rfm_segment |load_timestamp            |
+----------+----------------+--------------+-------------------+------------+---------+--------+-------+-------+-------+---------+------------+--------------------------+
|28389     |Rachael Martinez|France        |2011-05-31 00:00:00|1157        |1        |NULL    |4      |1      |1      |411      |New Customer|2026-05-12 14:24:24.012894|
|27601     |Sydney Rogers   |Southwest     |2011-06-04 00:00:00|1153        |1        |NULL    |4      |1      |1      |411      |New Customer|2026-05-12 14:24:24.012894|
|27612     |Lucas Hill      |Southwest     |2011-06-05 00:00:00|1152        |1        |NULL    |4      |1      |1      |41

## 7. Load — Simpan ke Hive

In [ ]:
spark.sql('CREATE DATABASE IF NOT EXISTS adventureworks_curated')

# saving to adventureworks_curated.fact_customer_rfm
df_final_rfm.write \
  .mode('overwrite') \
  .format('parquet') \
  .saveAsTable('adventureworks_curated.fact_customer_rfm')

# showing fact table & total rows
print('=== Verifikasi ===')
df_verify = spark.table('adventureworks_curated.fact_customer_rfm')
print(f'Total rows: {df_verify.count():,}')
df_verify.show(5)

=== Verifikasi ===
Total rows: 19,119
+----------+----------------+--------------+-------------------+------------+---------+--------+-------+-------+-------+---------+------------+--------------------+
|customerid|   customer_name|territory_name|    last_order_date|recency_days|frequency|monetary|r_score|f_score|m_score|rfm_score| rfm_segment|      load_timestamp|
+----------+----------------+--------------+-------------------+------------+---------+--------+-------+-------+-------+---------+------------+--------------------+
|     28389|Rachael Martinez|        France|2011-05-31 00:00:00|        1157|        1|    NULL|      4|      1|      1|      411|New Customer|2026-05-12 14:24:...|
|     27601|   Sydney Rogers|     Southwest|2011-06-04 00:00:00|        1153|        1|    NULL|      4|      1|      1|      411|New Customer|2026-05-12 14:24:...|
|     27612|      Lucas Hill|     Southwest|2011-06-05 00:00:00|        1152|        1|    NULL|      4|      1|      1|      411|New Cus

## 8. Analytic Queries

**Wajib: minimal 3 query yang menjawab business question**

In [20]:
# ── Query 1 (WAJIB): Top 20 Champion Customers ─────────────────────────────
# Tampilkan customer segment Champion, sort by monetary desc
# Kolom: customer_name, territory_name, recency_days, frequency, monetary, rfm_score

print('=== Query 1: Top 20 Champion Customers ===')
spark.sql("""
    SELECT
        customer_name,
        territory_name,
        recency_days,
        frequency,
        monetary,
        rfm_score
    FROM adventureworks_curated.fact_customer_rfm
    WHERE rfm_segment = 'Champion'
    ORDER BY monetary DESC
    LIMIT 20
""").show(truncate=False)

=== Query 1: Top 20 Champion Customers ===
+------------------+--------------+------------+---------+--------+---------+
|customer_name     |territory_name|recency_days|frequency|monetary|rfm_score|
+------------------+--------------+------------+---------+--------+---------+
|Maciej Dusza      |Northwest     |942         |2        |NULL    |433      |
|Shannon Elliott   |Southwest     |914         |2        |NULL    |433      |
|Stephen Mew       |Canada        |853         |2        |NULL    |433      |
|Elizabeth Keyser  |Canada        |853         |2        |NULL    |433      |
|Joshua Huff       |Southwest     |822         |2        |NULL    |433      |
|Charles Strange   |Southeast     |700         |2        |NULL    |433      |
|Jésus Hernandez   |Canada        |639         |2        |NULL    |433      |
|Esther Valle      |Canada        |608         |2        |NULL    |433      |
|Eddie Holmes      |Canada        |549         |2        |NULL    |433      |
|Adrian Dumitrascu |N

In [25]:
# ── Query 2 (WAJIB): Distribusi Segmen + Revenue Kontribusi ────────────────
# Tampilkan: rfm_segment, jumlah_customer, pct_customer, total_revenue, pct_revenue
# pct = % dari total

print('=== Query 2: Distribusi Segmen & Revenue Contribution ===')
spark.sql("""
    WITH segment_stats AS (
      SELECT
        rfm_segment,
        COUNT(customerid) AS jumlah_customer,
        SUM(monetary) AS total_revenue
      FROM adventureworks_curated.fact_customer_rfm
      GROUP BY rfm_segment
    )
    SELECT
        rfm_segment,
        jumlah_customer,
        ROUND(jumlah_customer * 100.0 / SUM(jumlah_customer) OVER(), 2) AS pct_customer,
        ROUND(total_revenue, 2) AS total_revenue,
        ROUND(total_revenue * 100.0 / SUM(total_revenue) OVER(), 2) AS pct_revenue
    FROM segment_stats
    ORDER BY total_revenue DESC
""").show()

=== Query 2: Distribusi Segmen & Revenue Contribution ===
+---------------+---------------+------------+-------------+-----------+
|    rfm_segment|jumlah_customer|pct_customer|total_revenue|pct_revenue|
+---------------+---------------+------------+-------------+-----------+
|          Loyal|           6494|       33.97|         NULL|       NULL|
|           Lost|            482|        2.52|         NULL|       NULL|
|       Champion|           3065|       16.03|         NULL|       NULL|
|         Others|           3824|       20.00|         NULL|       NULL|
|   New Customer|           3539|       18.51|         NULL|       NULL|
|Potential Loyal|           1715|        8.97|         NULL|       NULL|
+---------------+---------------+------------+-------------+-----------+



In [27]:
# ── Query 3 (WAJIB): At Risk & Lost Customers per Territory ────────────────
# Tampilkan pelanggan yang perlu di-winback
# Kolom: territory_name, rfm_segment, count, avg_monetary, avg_recency_days

print('=== Query 3: At Risk & Lost Customers per Territory ===')
spark.sql("""
    SELECT
      territory_name,
      rfm_segment,
      COUNT(customerid) AS customer_count,
      ROUND(AVG(monetary), 2) AS avg_monetary,
      ROUND(AVG(recency_days), 1) AS avg_recency_days
    FROM adventureworks_curated.fact_customer_rfm
    WHERE rfm_segment IN ('At Risk', 'Lost')
    GROUP BY territory_name, rfm_segment
    ORDER BY territory_name ASC, customer_count DESC
""").show(truncate=False)

=== Query 3: At Risk & Lost Customers per Territory ===
+--------------+-----------+--------------+------------+----------------+
|territory_name|rfm_segment|customer_count|avg_monetary|avg_recency_days|
+--------------+-----------+--------------+------------+----------------+
|Australia     |Lost       |30            |NULL        |108.6           |
|Canada        |Lost       |41            |NULL        |109.5           |
|Central       |Lost       |1             |NULL        |108.0           |
|France        |Lost       |57            |NULL        |109.7           |
|Germany       |Lost       |44            |NULL        |109.3           |
|Northeast     |Lost       |1             |NULL        |116.0           |
|Northwest     |Lost       |114           |NULL        |110.4           |
|Southeast     |Lost       |2             |NULL        |110.0           |
|Southwest     |Lost       |148           |NULL        |110.0           |
|United Kingdom|Lost       |44            |NULL        |

In [28]:
# ── Query 4 (BONUS): Average RFM metrics per Segment ──────────────────────
# Tampilkan profil rata-rata tiap segmen
# Kolom: rfm_segment, avg_recency, avg_frequency, avg_monetary, count_customers
# Sort: avg_monetary desc

print('=== Query 4 (Bonus): Profil Rata-rata per Segmen ===')
spark.sql("""
    SELECT
      rfm_segment,
      ROUND(AVG(recency_days), 1) AS avg_recency,
      ROUND(AVG(frequency), 2) AS avg_frequency,
      ROUND(AVG(monetary), 2) AS avg_monetary,
      COUNT(customerid) AS count_customers
    FROM adventureworks_curated.fact_customer_rfm
    GROUP BY rfm_segment
    ORDER BY avg_monetary DESC
""").show()

=== Query 4 (Bonus): Profil Rata-rata per Segmen ===
+---------------+-----------+-------------+------------+---------------+
|    rfm_segment|avg_recency|avg_frequency|avg_monetary|count_customers|
+---------------+-----------+-------------+------------+---------------+
|          Loyal|      102.0|         2.31|        NULL|           6494|
|           Lost|      109.8|          1.0|        NULL|            482|
|       Champion|      298.8|         2.26|        NULL|           3065|
|         Others|      193.6|          1.0|        NULL|           3824|
|   New Customer|      416.1|          1.0|        NULL|           3539|
|Potential Loyal|      225.5|          1.0|        NULL|           1715|
+---------------+-----------+-------------+------------+---------------+



In [30]:
# ── Query 5 (BONUS): Segmen Paling Banyak per Territory ───────────────────
# Untuk tiap territory, segmen mana yang paling dominan?
# Gunakan window function rank() partitionBy territory_name, orderBy desc count

print('=== Query 5 (Bonus): Dominant Segment per Territory ===')
spark.sql("""
    WITH segment_counts AS (
      SELECT
        territory_name,
        rfm_segment,
        COUNT(customerid) AS jumlah_customer
      FROM adventureworks_curated.fact_customer_rfm
      GROUP BY territory_name, rfm_segment
    ),
    ranked_segments AS (
      SELECT
        territory_name,
        rfm_segment,
        jumlah_customer,
        RANK() OVER (PARTITION BY territory_name ORDER BY jumlah_customer DESC) as rank
      FROM segment_counts
    )
    SELECT
      territory_name,
      rfm_segment AS dominant_segment,
      jumlah_customer
    FROM ranked_segments
    WHERE rank = 1
    ORDER BY jumlah_customer DESC
""").show(truncate=False)

=== Query 5 (Bonus): Dominant Segment per Territory ===
+--------------+----------------+---------------+
|territory_name|dominant_segment|jumlah_customer|
+--------------+----------------+---------------+
|Australia     |Loyal           |1707           |
|Southwest     |Loyal           |1200           |
|Northwest     |Loyal           |901            |
|United Kingdom|Loyal           |733            |
|Canada        |Loyal           |724            |
|Germany       |Loyal           |576            |
|France        |Loyal           |523            |
|Southeast     |Loyal           |52             |
|Northeast     |Loyal           |40             |
|Central       |Loyal           |38             |
+--------------+----------------+---------------+



In [31]:
spark.stop()
print('Done!')

Done!
